# Movie Recommendation System

This project is a simple movie recommendation system developed as part of Week 03 of the EncoderX AI/ML internship.

The main purpose of this project is to recommend movies based on the similarity between movies. The system uses movie information such as genres to find movies that are similar to a movie selected by the user.

In this project, the data is first cleaned and prepared. After that, a content based recommendation approach is used to generate movie recommendations.

Importing Required Libraries


In [4]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Loading the MovieLens Dataset

In [5]:
import requests
import zipfile
import io

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

response = requests.get(url, timeout=60)

print("Download status:", response.status_code)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall(".")

print("Dataset downloaded and extracted successfully!")

Download status: 200
Dataset downloaded and extracted successfully!


Reading the Dataset

In [6]:
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

print("Movies dataset shape:", movies.shape)
print("Ratings dataset shape:", ratings.shape)

Movies dataset shape: (9742, 3)
Ratings dataset shape: (100836, 4)


Exploring the Movies Dataset

In [7]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Exploring the Ratings Dataset

In [8]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Checking Dataset Information

In [9]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [10]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Checking Missing Values

In [11]:
print("Missing values in movies dataset:")
print(movies.isnull().sum())

print("\nMissing values in ratings dataset:")
print(ratings.isnull().sum())

Missing values in movies dataset:
movieId    0
title      0
genres     0
dtype: int64

Missing values in ratings dataset:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


Checking Duplicate Records

In [12]:
print("Duplicate rows in movies dataset:", movies.duplicated().sum())
print("Duplicate rows in ratings dataset:", ratings.duplicated().sum())

Duplicate rows in movies dataset: 0
Duplicate rows in ratings dataset: 0


Handling Missing Values

In [13]:
movies = movies.dropna(subset=["title", "genres"])

print("Missing values after cleaning:")
print(movies[["title", "genres"]].isnull().sum())

Missing values after cleaning:
title     0
genres    0
dtype: int64


Removing Duplicate Records

In [14]:
movies = movies.drop_duplicates()

print("Movies dataset shape after removing duplicates:", movies.shape)

Movies dataset shape after removing duplicates: (9742, 3)


Selecting Relevant Movie Information

In [15]:
movies = movies[["movieId", "title", "genres"]]

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Preparing Movie Genres

In [16]:
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


Preparing User Item Interactions

In [17]:
user_item_data = ratings[["userId", "movieId", "rating"]].copy()

user_item_data.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


Creating the User Item Interaction Matrix

In [18]:
user_item_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_item_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Final Data Check

In [19]:
print("Number of movies:", len(movies))
print("Number of ratings:", len(ratings))

movies.head(10)

Number of movies: 9742
Number of ratings: 100836


,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action Crime Thriller
6,7,Sabrina (1995),Comedy Romance
7,8,Tom and Huck (1995),Adventure Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action Adventure Thriller


## Recommendation Approach

A content-based filtering approach is used for this project.

The system recommends movies by comparing the genres of the selected movie with the genres of other movies. Movies with more similar genre information receive higher similarity scores and are recommended to the user.

This approach was selected because the movie dataset contains useful information about movie genres, which can be directly used to find similar movies.

Converting Movie Genres into Numerical Features

In [20]:
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(movies["genres"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9742, 24)


Calculating Movie Similarity

In [21]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (9742, 9742)


## Building the Recommendation Function

A recommendation function is created to find movies similar to a movie selected by the user.

The function compares the selected movie with other movies using the similarity scores calculated earlier. The movies with the highest similarity scores are returned as recommendations.

In [22]:
def recommend_movies(movie_title, num_recommendations=5):

    if movie_title not in movies["title"].values:
        return "Movie not found in the dataset."

    movie_index = movies[movies["title"] == movie_title].index[0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommended_movies = []

    for index, score in similarity_scores[1:num_recommendations + 1]:
        recommended_movies.append(movies.iloc[index]["title"])

    return recommended_movies

Testing the Recommendation System

In [23]:
movie_name = movies["title"].iloc[0]

print("Selected Movie:", movie_name)
print("\nRecommended Movies:")

recommend_movies(movie_name)

Selected Movie: Toy Story (1995)

Recommended Movies:


['Antz (1998)',
 'Toy Story 2 (1999)',
 'Adventures of Rocky and Bullwinkle, The (2000)',
 "Emperor's New Groove, The (2000)",
 'Monsters, Inc. (2001)']

In [24]:
movies.head(20)

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action Crime Thriller
6,7,Sabrina (1995),Comedy Romance
7,8,Tom and Huck (1995),Adventure Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action Adventure Thriller


In [25]:
recommend_movies("Toy Story (1995)", 5)

['Antz (1998)',
 'Toy Story 2 (1999)',
 'Adventures of Rocky and Bullwinkle, The (2000)',
 "Emperor's New Groove, The (2000)",
 'Monsters, Inc. (2001)']

Showing Recommendation Scores

In [26]:
def recommend_movies_with_scores(movie_title, num_recommendations=5):

    if movie_title not in movies["title"].values:
        return pd.DataFrame(columns=["Movie", "Similarity Score"])

    movie_index = movies[movies["title"] == movie_title].index[0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for index, score in similarity_scores[1:num_recommendations + 1]:
        results.append({
            "Movie": movies.iloc[index]["title"],
            "Similarity Score": round(score, 3)
        })

    return pd.DataFrame(results)

In [27]:
movie_name = "Toy Story (1995)"

print("Selected Movie:", movie_name)

recommend_movies_with_scores(movie_name, 5)

Selected Movie: Toy Story (1995)


,Movie,Similarity Score
0,Antz (1998),1.0
1,Toy Story 2 (1999),1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",1.0
3,"Emperor's New Groove, The (2000)",1.0
4,"Monsters, Inc. (2001)",1.0


Evaluation Using Precision@5

In [28]:
def precision_at_k(movie_title, k=5):

    if movie_title not in movies["title"].values:
        return 0

    movie_index = movies[movies["title"] == movie_title].index[0]

    selected_genres = set(movies.iloc[movie_index]["genres"].split())

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_k = similarity_scores[1:k + 1]

    relevant_count = 0

    for index, score in top_k:
        recommended_genres = set(movies.iloc[index]["genres"].split())

        if selected_genres.intersection(recommended_genres):
            relevant_count += 1

    return relevant_count / k

Precision@5 Result

In [29]:
test_movie = "Toy Story (1995)"

precision_score = precision_at_k(test_movie, 5)

print("Test Movie:", test_movie)
print("Precision@5:", round(precision_score, 2))

Test Movie: Toy Story (1995)
Precision@5: 1.0


Testing Multiple Movies

In [30]:
test_movies = movies["title"].head(10).tolist()

scores = []

for movie in test_movies:
    score = precision_at_k(movie, 5)
    scores.append(score)

evaluation_results = pd.DataFrame({
    "Movie": test_movies,
    "Precision@5": scores
})

evaluation_results

,Movie,Precision@5
0,Toy Story (1995),1.0
1,Jumanji (1995),1.0
2,Grumpier Old Men (1995),1.0
3,Waiting to Exhale (1995),1.0
4,Father of the Bride Part II (1995),1.0
5,Heat (1995),1.0
6,Sabrina (1995),1.0
7,Tom and Huck (1995),1.0
8,Sudden Death (1995),1.0
9,GoldenEye (1995),1.0


Average Precision@5

In [31]:
average_precision = evaluation_results["Precision@5"].mean()

print("Average Precision@5:", round(average_precision, 2))

Average Precision@5: 1.0


Evaluation Summary

In [32]:
print("Number of movies tested:", len(evaluation_results))
print("Average Precision@5:", round(average_precision, 2))

Number of movies tested: 10
Average Precision@5: 1.0


Demonstration Interface

In [33]:
!pip -q install gradio

Creating the Recommendation Interface

In [34]:
import gradio as gr

def get_recommendations(movie_title, number_of_movies):
    result = recommend_movies_with_scores(
        movie_title,
        int(number_of_movies)
    )

    if result.empty:
        return "Movie not found. Please enter an exact movie title from the dataset."

    return result.to_string(index=False)


interface = gr.Interface(
    fn=get_recommendations,
    inputs=[
        gr.Textbox(
            label="Enter Movie Title",
            placeholder="Example: Toy Story (1995)"
        ),
        gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="Number of Recommendations"
        )
    ],
    outputs=gr.Textbox(label="Recommended Movies"),
    title="Movie Recommendation System",
    description="Enter a movie title to get similar movie recommendations."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://06862a499ddeca112f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Project Overview

This project is a movie recommendation system made for Week 03 of the EncoderX AI/ML internship.

The main purpose of this project is to recommend movies that are similar to a movie selected by the user.

For this project, I used the MovieLens dataset and built the recommendation system using content-based filtering. The system checks the genres of movies and finds other movies with similar genres.

## Dataset

I used the MovieLens dataset for this project.

The dataset contains information about movies and user ratings. The movie file contains the movie ID, movie title, and genres. The ratings file contains user ID, movie ID, rating, and timestamp.

The dataset was downloaded and used directly in Google Colab.

## Data Preparation

Before building the recommendation system, I checked and cleaned the data.

The main steps I followed were:

- Checked for missing values
- Removed records where the movie title or genre was missing
- Checked for duplicate records
- Selected the columns needed for the project
- Cleaned the movie genre information
- Prepared the user and movie interaction data
- Created a user-item matrix

## Recommendation Method

I used content-based filtering for this project.

First, the movie genres were converted into numerical values using TF-IDF. After that, cosine similarity was used to compare the movies.

When a user enters a movie, the system checks which other movies have similar genre information and shows the most similar movies as recommendations.

## Evaluation

I used Precision@5 to check the performance of the recommendation system.

For testing, I used 10 movies. A movie was counted as relevant if it had at least one genre in common with the selected movie.

The average Precision@5 result was:

**1.0**

This result is based on the relevance condition used in this project.

## Demonstration Interface

I created a simple interface using Gradio so the recommendation system can be tested easily.

The user can enter a movie title and choose how many recommendations they want. The system then shows the recommended movies along with their similarity scores.

For example, I tested the system with "Toy Story (1995)" and it returned five similar movies.

## Conclusion

In this project, I built a simple movie recommendation system using the MovieLens dataset.

The system uses content-based filtering to find movies with similar genres. I cleaned the data, prepared the movie information, converted the genres into numerical features using TF-IDF, and used cosine similarity to find similar movies.

I also tested the recommendation system with different movies and evaluated it using Precision@5. Ten movies were tested and the average Precision@5 was 1.0 based on the relevance condition used in this project.

Finally, I created a simple Gradio interface where a user can enter a movie name and get recommendations.

## Tools and Technologies Used

- Python
- Google Colab
- Pandas
- NumPy
- Scikit-learn
- TF-IDF
- Cosine Similarity
- Gradio
- MovieLens Dataset